# Special Methods and Operator Overloading
---
# Special or Magic or Dunder Methods

* Special, _Magic_, or _Dunder_, methods are special methods within Python associated with an object

* The term "dunder" comes from "double underscore", which is a characteristic of these methods
    * __init__
    * __str__
    * __len__
    * etc.
* There are many special methods in Python that are at the core of Python and how it supports its object-oriented features

---
# Under the Hood
* Many operations within Python implicitly call magic methods to execute certain operations
* These methods are not intended to be directly called by you, but you can override them as we'll see later.

In [ ]:
3 + 4

Under the hood this calls

In [ ]:
(3).__add__(4)

---
# A Few Magic Methods

A handful you'll meet repeatedly:

| Dunder       | Triggers when                          |
| :----------- | :------------------------------------- |
| `__init__`   | An instance is constructed             |
| `__str__`    | `str(obj)` or `print(obj)` is called   |
| `__repr__`   | The interpreter displays the object    |
| `__len__`    | `len(obj)` is called                   |
| `__add__`    | `obj + other` is evaluated             |
| `__eq__`     | `obj == other` is evaluated            |
| `__call__`   | `obj(...)` is called                   |

# Controlling the Object Creation Process

* When you call a class constructer you create a new instance of that class
* When this happens Python invokes the `__new__()` method as the first step
  * This method is responsible for creating and returning a new empty object of this class
* This new object is then passed to `__init__()` to initialize the object with the appropriate values and properties
  * If you're familiar with OOP concepts, this is the Contstructor
  * Remember, all methods talk a first argument traditionally named `self`

---
# `__init__()` Example

In [ ]:
class Person:

    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

mason = Person("Mason", "Egger")
print(mason)

---
# Representing Objects as Strings

To represent the object as a human-readable string instead of the object reference, implement the `__str__()` method.

In [ ]:
class Person:

    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    def __str__(self):
      return f"{self.first_name} {self.last_name}"

mason = Person("Mason", "Egger")
print(mason)

---
# Making Your Objects Callable with `__call__`

* You can implement the `__call__()` method to make an instance callable after creation
* Sometimes this is the right tool. Sometimes it isn't. Let's see what it does, then look at why it's controversial.


In [ ]:
class Factorial:
    def __init__(self):
        self._cache = {0: 1, 1: 1}

    def __call__(self, number):
        if number not in self._cache:
            self._cache[number] = number * self(number - 1)
        return self._cache[number]

factorial = Factorial()

print(factorial(4))
print(factorial(5))
print(factorial(6))

**Considered by some to be an antipattern, and here's why:**

* `factorial(5)` reads like a function call, but the instance is hiding state in `_cache`
* Readers can't tell from the call site that there's caching going on
* A function with `functools.lru_cache` does the same thing more clearly:

```python
import functools

@functools.lru_cache
def factorial(n):
    return 1 if n < 2 else n * factorial(n - 1)
```

* Reach for `__call__` only when you genuinely want a stateful object that *acts* like a function (configured strategies, pre-bound clients). Otherwise, prefer functions or closures.


---
# Operator Overloading



---
# Operator Overloading

Operator overloading is redefining the behavior of built-in operators for use with user-defined classes in Python.

This is a _very_ powerful feature in programming languages and can easily lead to confusion and errors. Use caution when overloading operators.

A few things to remember:
* Cannot overload operators for the built-in types
* Cannot create new operators, only overload existing ones
* A few operators can't be overloaded
  * `is`, `and`, `or`, `not`
    * Although the bitwise operators can be

---
# Overriding Mathematical Operators

* The operators `+`, `-`, `*`, `/`, etc. can be overridden for use with your custom object

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        new_x = self.x + other.x
        new_y = self.y + other.y
        return Point(new_x, new_y)

    def __str__(self):
        return f"Point ({self.x}, {self.y})"



x = Point(1, 2)
y = Point(3, 4)

x + y

---
# Wait, why didn't my string get printed?

* `__str__()` produces a nice, human readable format when the object is being requested as a string, such as in a print statement
* `__repr__()` is more for developers. It is an unambiguious stirng representation and will be interprested by the interpreter correctly. It should list enough information that you are able to recreate the object from it.
* When in doubt, implement both

---
# `__repr__` in Action

Implementing both `__str__` and `__repr__` lets you control how the object appears in different contexts.

In [ ]:
class Person:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    def __str__(self):
        return f"{self.first_name} {self.last_name}"

    def __repr__(self):
        return f"Person({self.first_name!r}, {self.last_name!r})"

mason = Person("Mason", "Egger")
print(mason)         # __str__:  Mason Egger
print(repr(mason))   # __repr__: Person('Mason', 'Egger')
print([mason])       # Lists call __repr__ on their items

---
# Overloading Comparison Operators

* You can also overload comparison operaters such as `==`, `!=`, `>`, `<`, `<=`, etc.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if self.x == other.x and self.y == other.y:
            return True
        return False

x = Point(1, 2)
y = Point(3, 4)
z = Point(1,2)

print(x == y)
print(x == z)
print(x == x)
print(z == x)

---
# Container Protocols: `__len__` and `__getitem__`

Implement these and your class behaves like a sequence. `len()`, indexing, and iteration all work for free.

In [ ]:
class Deck:
    def __init__(self, cards):
        self._cards = list(cards)

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, index):
        return self._cards[index]

deck = Deck(["Ace", "King", "Queen", "Jack"])
print(len(deck))    # 4
print(deck[0])      # Ace
print(deck[-1])     # Jack
for card in deck:   # Iteration works for free
    print(card)

---
# And so much more!

We only scratched the surface of special methods. There are 80+ special methods that allow you to control nearly every aspect of your objects. Visit the [Python Documentation](https://docs.python.org/3/reference/datamodel.html#specialnames) to learn about more.

---
# Summary
* Special, _Magic_, or _Dunder_, methods are special methods within Python associated with an object
* The term "dunder" comes from "double underscore", which is a characteristic of these methods
* There are many magic methods in Python that are at the core of Python and how it supports its object-oriented features
* Many operations within Python implicitly call magic methods to execute certain operations
* These methods are not intended to be directly called by you, but you can override them to modify the functionality.

---
# Exercise 4 - Special Methods
* In these exercises you will implement a Stack using only special methods
* You'll know you've succeeded when:
  * `stack + N` pushes `N` onto the stack
  * `stack - 1` pops and returns the last item, raising `IndexError` on an empty stack
  * `print(stack)` shows the current contents
  * (Stretch) `stack[i]` and `len(stack)` work
* Go to the Exercise Directory in the Google Drive and open the Practice Directory
* Open _04-Special-Methods-and-Operator-Overloading.ipynb_ and follow the instructions
* If you get stuck, raise your hand and someone will come by and help. You can also check the Solution directory for the answers
* You have **15 mins**
